In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np

In [2]:
df = pd.read_csv('https://drive.google.com/uc?id=15RfMD9lNkpS3cVN7j3_dsJKZ8_5RJG5z')
df.head()

,date,meantemp,humidity,wind_speed,meanpressure
0,2013-01-01,10.000000,84.500000,0.000000,1015.666667
1,2013-01-02,7.400000,92.000000,2.980000,1017.800000
2,2013-01-03,7.166667,87.000000,4.633333,1018.666667
3,2013-01-04,8.666667,71.333333,1.233333,1017.166667
4,2013-01-05,6.000000,86.833333,3.700000,1016.500000


1. **def windowed_dataset(series, window_size, batch_size, shuffle_buffer**):ini adalah definisi fungsi windowed_dataset yang menerima empat parameter:
- **series**: data deret waktu.
- **window_size**: ukuran jendela (window) yang akan digunakan.
- **batch_size**: ukuran batch.
- **shuffle_buffer**: ukuran buffer untuk pengacakan (shuffle).
2. **series = tf.expand_dims(series, axis=-1)**: mengubah dimensi series dengan menambahkan satu dimensi baru di posisi terakhir (axis=-1). Ini biasanya dilakukan untuk menangani data deret waktu yang memiliki satu dimensi (misalnya, satu fitur. Pada kasus ini meantemp).
3. **ds = tf.data.Dataset.from_tensor_slices(series)**: membuat dataset TensorFlow menjadi sebuah series. Dataset ini berisi slice dari series dimana setiap slice sesuai dengan satu elemen dalam series.
4. **ds = ds.window(window_size + 1, shift=1, drop_remainder=True)**: membagi dataset menjadi jendela-jendela berukuran window_size + 1 dengan jarak 1 (shift=1) antara jendela-jendela tersebut. drop_remainder=True menghapus jendela-jendela yang tidak cukup panjang untuk membentuk satu jendela dengan ukuran yang diinginkan.
5. **ds = ds.flat_map(lambda w: w.batch(window_size + 1))**: mengubah setiap window menjadi sebuah batch dengan ukuran window_size + 1. Ini dilakukan menggunakan flat_map yang menjalankan fungsi yang diberikan pada setiap elemen dataset dan kemudian menggabungkan hasilnya.
6. **ds = ds.shuffle(shuffle_buffer)**: mengacak dataset dengan menggunakan buffer ukuran shuffle_buffer. Ini akan mengacak elemen-elemen dataset, yang berguna untuk memperkenalkan variasi dalam pelatihan model.
7. **ds = ds.map(lambda w: (w[:-1], w[-1:]))**: memetakan setiap batch menjadi sepasang data input dan target. Data input adalah semua elemen dalam batch kecuali elemen terakhir, sedangkan targetnya adalah elemen terakhir. Ini digunakan untuk melatih model untuk memprediksi elemen berikutnya dalam deret waktu berdasarkan elemen-elemen sebelumnya.
8. **return ds.batch(batch_size).prefetch(1)**: mengelompokkan batch-batch dataset menjadi batch dengan ukuran batch_size, dan menambahkan prefetching ke dataset. Prefetching memungkinkan TensorFlow untuk memuat data selanjutnya ke dalam memori ketika model sedang melakukan komputasi sehingga dapat meningkatkan kinerja. Dengan nilai argumen 1, prefetch akan memastikan bahwa satu batch selalu siap untuk diproses oleh model.

In [3]:
def windowed_dataset(series, window_size, batch_size, shuffle_buffer):
  series = tf.expand_dims(series, axis=-1)
  ds = tf.data.Dataset.from_tensor_slices(series)
  ds = ds.window(window_size + 1, shift=1, drop_remainder=True)
  ds = ds.flat_map(lambda w: w.batch(window_size + 1))
  ds = ds.shuffle(shuffle_buffer)
  ds = ds.map(lambda w: (w[:-1], w[-1:]))
  return ds.batch(batch_size).prefetch(1)

menentukan fitur yang akan digunakan

In [5]:
dates = df['date'].values
temp = df['meantemp'].values

menggunakan fungsi windowing

In [6]:
train_set = windowed_dataset(temp, window_size=2, batch_size=3, shuffle_buffer=1000)

tanpa tf

In [7]:
for data in train_set:
 print(data)
 break

(<tf.Tensor: shape=(3, 2, 1), dtype=float64, numpy=
array([[[16.625],
        [17.75 ]],

       [[24.6  ],
        [25.6  ]],

       [[17.75 ],
        [17.875]]])>, <tf.Tensor: shape=(3, 1, 1), dtype=float64, numpy=
array([[[12.875     ]],

       [[25.85714286]],

       [[18.        ]]])>)
